# 365 Probabilidades - Dia #117
## Qual a probabilidade de o time da casa ganhar?

**Tipo:** Probabilidade simples
**Data de publicação:** 2026-10-08
**Ferramenta:** Python
**Decisão analisada:** Quanto vale jogar em casa, e quanto disso é a torcida?
**Hashtag:** #365Probabilidades #Dia117

---

### 📖 A História

Todo mundo já marcou um jogo em casa como "esse a gente ganha".

Casa cheia, time descansado, o adversário vindo de longe. A conta parece feita antes
de a bola rolar. Aí o jogo acaba 1 a 1 e a explicação vem pronta: foi azar.

Eu quis saber quanto a casa vale de verdade. Não a impressão, o número. Quantos jogos
o mandante ganha, há quanto tempo isso é assim, e o que aconteceu quando o estádio
ficou vazio.

---

### 📚 O Conceito

Vantagem de jogar em casa é a diferença entre o que o mandante consegue e o que ele
conseguiria num campo neutro. Como o campo neutro não existe no campeonato, a medida
usual é simples: a fração dos jogos (ou dos pontos) que fica com quem joga em casa.

A literatura separa quatro causas: torcida, viagem, familiaridade com o campo e
arbitragem. Elas andam juntas e são difíceis de separar. A pandemia separou uma delas
à força: em 2020, o Brasileirão inteiro foi jogado sem público.

---

### 🧮 O Modelo

Este é um dia de **probabilidade simples**: a pergunta vem primeiro e o dado é registro
de placar, não artigo. A conta é toda nossa, e dois artigos entram como régua de
comparação.

**Dados:**
- Resultados da Série A do Campeonato Brasileiro, 2003 a 2024, **8.785 jogos**, com
  mandante, visitante, placar, estádio e estado de cada clube. Base pública compilada
  a partir dos registros da CBF (repositório `adaoduque/Brasileirao_Dataset`, arquivo
  `campeonato-brasileiro-full.csv`). Conferida aqui: número de jogos por temporada,
  datas, placares e coerência do vencedor com o placar.

**Fontes de comparação:**
- Pollard, R., Silva, C. D. & Medeiros, N. C. (2008). *Home advantage in football in
  Brazil: differences between teams and the effects of distance traveled.* Brazilian
  Journal of Soccer Science 1(1), 3-10. Série A de 2003 a 2007, 2.326 jogos: 65,2% dos
  pontos com o mandante; 0,115 gol a favor da casa por 1.000 km viajados pelo visitante.
- Bryson, A., Dolton, P., Reade, J. J., Schreyer, D. & Singleton, C. (2021). *Causal
  effects of an absent crowd on performances and refereeing decisions during Covid-19.*
  Economics Letters 198, 109664. 6.481 jogos em 23 ligas de 17 países, 1.498 de portões
  fechados: um terço a menos de cartões amarelos para o visitante; sem efeito
  significativo em vitórias, gols ou pontos.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE = 42
ger = np.random.default_rng(SEMENTE)

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=1):
    return vir(100 * x, casas) + "%"

def mil(n):
    return f"{int(n):,}".replace(",", ".")

print("✅ Bibliotecas carregadas")

In [ ]:
# --- OS DADOS ---
# Serie A do Brasileirao, 2003 a 2024. Base publica compilada dos registros da CBF.
# Antes de qualquer conta, a base passa por quatro conferencias.

d = pd.read_csv("campeonato-brasileiro-full.csv")
d["dt"] = pd.to_datetime(d["data"], dayfirst=True)
d["gm"] = d["mandante_Placar"].astype(int)
d["gv"] = d["visitante_Placar"].astype(int)

# Temporada: a edicao de 2020 foi jogada de agosto/2020 a fevereiro/2021.
# Jogos de janeiro e fevereiro de 2021 pertencem a 2020, nao a 2021.
d["temporada"] = d["dt"].dt.year
d.loc[(d["dt"] >= "2021-01-01") & (d["dt"] < "2021-04-01"), "temporada"] = 2020

# Resultado a partir do PLACAR, nao da coluna "vencedor" da base.
d["res"] = np.select([d["gm"] > d["gv"], d["gm"] < d["gv"]], ["casa", "fora"], "empate")
venc_base = np.where(d["gm"] > d["gv"], d["mandante"],
                     np.where(d["gm"] < d["gv"], d["visitante"], "-"))
inconsistentes = int((venc_base != d["vencedor"]).sum())

por_temp = d.groupby("temporada").agg(jogos=("ID", "size"), clubes=("mandante", "nunique"),
                                      inicio=("dt", "min"), fim=("dt", "max"))
esperado = por_temp["clubes"] * (por_temp["clubes"] - 1)
faltam = (esperado - por_temp["jogos"])

print("=" * 72)
print("  CONFERÊNCIA DA BASE")
print("=" * 72)
print(f"\n  Jogos: {mil(len(d))}   temporadas: {d['temporada'].nunique()}"
      f"   de {d['dt'].min():%d/%m/%Y} a {d['dt'].max():%d/%m/%Y}")
print(f"  Placares faltando: {int(d[['gm', 'gv']].isna().sum().sum())}"
      f"   duplicatas (data, mandante, visitante): "
      f"{int(d.duplicated(['data', 'mandante', 'visitante']).sum())}")
print(f"  Linhas em que a coluna 'vencedor' discorda do placar: {inconsistentes}"
      f"  (o placar manda; a linha é de 2005, Brasiliense 0 x 1 marcado como empate)")
print(f"\n  Jogos por temporada contra o esperado (clubes × (clubes − 1)):")
for t, row in por_temp.iterrows():
    marca = "" if faltam[t] == 0 else f"   ← faltam {int(faltam[t])}"
    print(f"    {t}: {row['jogos']:>3} jogos, {row['clubes']} clubes{marca}")
print(f"\n  2016 tem 379 e não 380: o jogo Chapecoense × Atlético-MG não foi disputado")
print(f"  (acidente aéreo de novembro de 2016). Não é falha da base.")
print(f"  2003 (24 clubes) e 2005 (22 clubes) têm fórmulas diferentes de 2006 em diante.")

print("\n" + "=" * 72)
print("  O NÚMERO BRUTO")
print("=" * 72)
freq = d["res"].value_counts(normalize=True)
print(f"\n  Em {mil(len(d))} jogos:")
print(f"    time da casa vence   {pct(freq['casa'])}")
print(f"    empate               {pct(freq['empate'])}")
print(f"    visitante vence      {pct(freq['fora'])}")
print(f"\n  ×0,80: não se aplica. Registro de placar, não autorrelato.")
print("=" * 72)

In [ ]:
# --- O MODELO ---
# Quatro camadas, todas conta propria.

def beta_j(x, n):
    # posterior de Jeffreys para uma proporcao: Beta(x + 0,5; n - x + 0,5)
    return stats.beta(x + 0.5, n - x + 0.5)

def resumo(dist):
    lo, hi = dist.interval(0.95)
    return dist.mean(), lo, hi

n_tot = len(d)
x_casa = int((d["res"] == "casa").sum())
post_casa = beta_j(x_casa, n_tot)
m, lo, hi = resumo(post_casa)

print("=" * 72)
print("  CAMADA 1 · A PROBABILIDADE, COM INTERVALO")
print("=" * 72)
print(f"\n  P(casa vence) = {pct(m)}   intervalo de credibilidade de 95%: [{pct(lo)}; {pct(hi)}]")
print(f"  Com {mil(n_tot)} jogos o intervalo tem dois pontos de largura. A pergunta do dia")
print(f"  está respondida: um pouco menos da metade. O resto do notebook pergunta")
print(f"  o que está dentro desse número.")

# por temporada
tab = d.groupby("temporada")["res"].value_counts().unstack(fill_value=0)
tab["n"] = tab.sum(axis=1)
tab["p_casa"] = tab["casa"] / tab["n"]
ic = np.array([beta_j(x, n).interval(0.95) for x, n in zip(tab["casa"], tab["n"])])
tab["lo"], tab["hi"] = ic[:, 0], ic[:, 1]
# fracao dos pontos distribuidos que ficou com o mandante (vitoria 3, empate 1 para cada lado)
tab["pontos_casa"] = (3 * tab["casa"] + tab["empate"]) / (3 * (tab["casa"] + tab["fora"]) + 2 * tab["empate"])

print(f"\n  {'temporada':>10}{'jogos':>7}{'casa':>8}{'IC 95%':>18}{'pontos c/ casa':>16}")
for t, r in tab.iterrows():
    print(f"  {t:>10}{int(r['n']):>7}{pct(r['p_casa']):>8}"
          f"   [{pct(r['lo'])}; {pct(r['hi'])}]{pct(r['pontos_casa']):>14}")

# ── CAMADA 2 · validacao contra Pollard 2008 ──
sub = d[(d["temporada"] >= 2003) & (d["temporada"] <= 2007)]
pts_casa = (3 * (sub["res"] == "casa").sum() + (sub["res"] == "empate").sum())
pts_tot = 3 * (sub["res"] != "empate").sum() + 2 * (sub["res"] == "empate").sum()
share_pollard = pts_casa / pts_tot
print("\n" + "=" * 72)
print("  CAMADA 2 · A BASE CONTRA O ARTIGO (2003 a 2007)")
print("=" * 72)
print(f"\n  Pollard, Silva & Medeiros 2008: 2.326 jogos, 65,24% dos pontos com o mandante.")
print(f"  Esta base, mesmas temporadas: {mil(len(sub))} jogos, {pct(share_pollard, 2)} dos pontos.")
print(f"  Diferença: {vir((share_pollard - 0.6524) * 100, 2)} ponto percentual.")
print(f"  A base reproduz o artigo. É a conferência externa que autoriza o resto.")

# ── CAMADA 3 · o experimento natural ──
def regime(t):
    if t <= 2019: return "com público (2003 a 2019)"
    if t == 2020: return "sem público (2020)"
    if t == 2021: return "transição (2021)"
    return "público de volta (2022 a 2024)"
d["regime"] = d["temporada"].map(regime)
ordem = ["com público (2003 a 2019)", "sem público (2020)", "transição (2021)",
         "público de volta (2022 a 2024)"]

print("\n" + "=" * 72)
print("  CAMADA 3 · O EXPERIMENTO NATURAL: 2020 SEM PÚBLICO")
print("=" * 72)
print(f"\n  {'regime':<32}{'jogos':>7}{'casa':>8}{'empate':>9}{'fora':>8}{'IC casa':>20}")
reg = {}
for r in ordem:
    s = d[d["regime"] == r]
    n = len(s); xc = int((s["res"] == "casa").sum())
    xe = int((s["res"] == "empate").sum()); xf = n - xc - xe
    reg[r] = dict(n=n, casa=xc, empate=xe, fora=xf, post=beta_j(xc, n))
    lo, hi = reg[r]["post"].interval(0.95)
    print(f"  {r:<32}{n:>7}{pct(xc / n):>8}{pct(xe / n):>9}{pct(xf / n):>8}"
          f"   [{pct(lo)}; {pct(hi)}]")

N_SIM = 200_000
a = reg["com público (2003 a 2019)"]["post"].rvs(N_SIM, random_state=ger)
b = reg["sem público (2020)"]["post"].rvs(N_SIM, random_state=ger)
c = reg["público de volta (2022 a 2024)"]["post"].rvs(N_SIM, random_state=ger)
dif_ab = a - b; dif_ac = a - c; dif_cb = c - b
def ic_sim(v):
    return np.mean(v), np.percentile(v, 2.5), np.percentile(v, 97.5)
for nome, v in [("com público (2003-2019) − sem público (2020)", dif_ab),
                ("com público (2003-2019) − público de volta (2022-2024)", dif_ac),
                ("público de volta (2022-2024) − sem público (2020)", dif_cb)]:
    m_, lo_, hi_ = ic_sim(v)
    print(f"\n  {nome}:")
    print(f"    diferença {vir(m_ * 100)} pp   IC 95% [{vir(lo_ * 100)}; {vir(hi_ * 100)}]"
          f"   P(diferença > 0) = {pct(np.mean(v > 0))}")

# Sensibilidade: a vantagem da casa ja vinha caindo antes de 2020. Linha de base recente.
for ini, fim in [(2015, 2019), (2017, 2019)]:
    s_ = d[(d["temporada"] >= ini) & (d["temporada"] <= fim)]
    p_ = beta_j(int((s_["res"] == "casa").sum()), len(s_)).rvs(N_SIM, random_state=ger)
    m_, lo_, hi_ = ic_sim(p_ - b)
    print(f"\n  Linha de base {ini} a {fim} ({mil(len(s_))} jogos, {pct(np.mean(p_))}) − sem público (2020):")
    print(f"    diferença {vir(m_ * 100)} pp   IC 95% [{vir(lo_ * 100)}; {vir(hi_ * 100)}]"
          f"   P(diferença > 0) = {pct(np.mean(p_ - b > 0))}")
print(f"\n  2017, com público, teve {pct(tab.loc[2017, 'p_casa'])}: uma temporada sozinha oscila")
print(f"  cinco pontos para cada lado. O que sustenta o efeito é a comparação com")
print(f"  milhares de jogos, não o ranking de uma temporada.")

# Pontos por jogo em casa, com Dirichlet (Jeffreys) sobre os tres desfechos
def pontos_casa_dist(xc, xe, xf, n_sim=N_SIM):
    p = ger.dirichlet([xc + 0.5, xe + 0.5, xf + 0.5], size=n_sim)
    return 3 * p[:, 0] + p[:, 1]
ppj = {r: pontos_casa_dist(reg[r]["casa"], reg[r]["empate"], reg[r]["fora"]) for r in ordem}
print(f"\n  Pontos esperados do mandante por jogo em casa:")
for r in ordem:
    m_, lo_, hi_ = ic_sim(ppj[r])
    print(f"    {r:<32}{vir(m_, 2)}   [{vir(lo_, 2)}; {vir(hi_, 2)}]")
dif_pts = ppj["com público (2003 a 2019)"] - ppj["sem público (2020)"]
m_, lo_, hi_ = ic_sim(dif_pts)
print(f"\n  Em 19 jogos em casa, a diferença entre os dois regimes vale "
      f"{vir(19 * m_)} pontos por temporada [{vir(19 * lo_)}; {vir(19 * hi_)}].")
print(f"  2021 fica de fora das comparações: o público voltou aos poucos, estádio a")
print(f"  estádio, e a base não diz quem jogou com e quem jogou sem torcida.")

# ── CAMADA 4 · a distancia ──
# Sem a cidade de cada clube, a distancia e aproximada pela capital do estado.
# Jogos entre clubes do mesmo estado entram com distancia zero.
cap = {"SP": (-23.55, -46.63), "RJ": (-22.91, -43.17), "MG": (-19.92, -43.94),
       "RS": (-30.03, -51.23), "PR": (-25.43, -49.27), "SC": (-27.60, -48.55),
       "BA": (-12.97, -38.51), "PE": (-8.05, -34.88), "CE": (-3.72, -38.54),
       "GO": (-16.68, -49.25), "DF": (-15.79, -47.88), "MT": (-15.60, -56.10),
       "PA": (-1.46, -48.50), "AL": (-9.67, -35.74), "RN": (-5.79, -35.21)}
def haversine(a, b):
    la1, lo1 = np.radians(a); la2, lo2 = np.radians(b)
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(h))
d["km"] = [0.0 if m == v else haversine(cap[m], cap[v])
           for m, v in zip(d["mandante_Estado"], d["visitante_Estado"])]
d["saldo"] = d["gm"] - d["gv"]

print("\n" + "=" * 72)
print("  CAMADA 4 · A VIAGEM DO VISITANTE")
print("=" * 72)
faixas = [(-1, 0, "mesmo estado"), (0, 1000, "até 1.000 km"),
          (1000, 2000, "1.000 a 2.000 km"), (2000, 9999, "mais de 2.000 km")]
print(f"\n  {'distância':<20}{'jogos':>7}{'casa':>8}{'IC 95%':>18}{'saldo médio':>13}")
faixa_res = []
for lo_k, hi_k, nome in faixas:
    s = d[(d["km"] > lo_k) & (d["km"] <= hi_k)]
    n = len(s); xc = int((s["res"] == "casa").sum())
    lo, hi = beta_j(xc, n).interval(0.95)
    faixa_res.append((nome, n, xc / n, lo, hi, s["saldo"].mean()))
    print(f"  {nome:<20}{n:>7}{pct(xc / n):>8}   [{pct(lo)}; {pct(hi)}]{vir(s['saldo'].mean(), 2):>13}")

# Regressao linear do saldo de gols sobre a distancia (em mil km), como Pollard.
X = d["km"].values / 1000; Y = d["saldo"].values
b1, b0 = np.polyfit(X, Y, 1)
B = 5000
coefs = np.empty(B)
for i in range(B):
    idx = ger.integers(0, len(X), len(X))
    coefs[i] = np.polyfit(X[idx], Y[idx], 1)[0]
lo_b, hi_b = np.percentile(coefs, [2.5, 97.5])
print(f"\n  Saldo de gols do mandante por 1.000 km de viagem do visitante:")
print(f"    esta base (2003 a 2024, capitais como proxy): {vir(b1, 3)} gol"
      f"   IC 95% bootstrap [{vir(lo_b, 3)}; {vir(hi_b, 3)}]")
print(f"    Pollard, Silva & Medeiros 2008 (2003 a 2007, cidades): 0,115 gol")
print(f"  Mesma ordem de grandeza, mesmo sinal. A distância é uma parte pequena da")
print(f"  vantagem: o mandante já ganha {pct(faixa_res[0][2])} dos clássicos estaduais,")
print(f"  em que ninguém viaja.")

# ── CAMADA 5 · a temporada simulada ──
print("\n" + "=" * 72)
print("  CAMADA 5 · O QUE A TORCIDA VALE NUMA TEMPORADA")
print("=" * 72)
def temporada_pontos(xc, xe, xf, n_sim=N_SIM, jogos=19):
    p = ger.dirichlet([xc + 0.5, xe + 0.5, xf + 0.5], size=n_sim)
    sorte = ger.random((n_sim, jogos))
    vit = (sorte < p[:, [0]]).sum(axis=1)
    emp = ((sorte >= p[:, [0]]) & (sorte < p[:, [0]] + p[:, [1]])).sum(axis=1)
    return 3 * vit + emp
pts_com = temporada_pontos(reg["com público (2003 a 2019)"]["casa"],
                           reg["com público (2003 a 2019)"]["empate"],
                           reg["com público (2003 a 2019)"]["fora"])
pts_sem = temporada_pontos(reg["sem público (2020)"]["casa"], reg["sem público (2020)"]["empate"],
                           reg["sem público (2020)"]["fora"])
dif_temp = pts_com - pts_sem
print(f"\n  Um clube médio, 19 jogos em casa, {mil(N_SIM)} temporadas sorteadas:")
print(f"    com público:  {vir(pts_com.mean())} pontos em casa   [{int(np.percentile(pts_com, 2.5))}; {int(np.percentile(pts_com, 97.5))}]")
print(f"    sem público:  {vir(pts_sem.mean())} pontos em casa   [{int(np.percentile(pts_sem, 2.5))}; {int(np.percentile(pts_sem, 97.5))}]")
print(f"    diferença:    {vir(dif_temp.mean())} pontos   P(perde pontos sem público) = {pct(np.mean(dif_temp > 0))}")
print(f"  A diferença entre regimes é pequena diante do acaso de uma temporada:")
print(f"  o intervalo de um clube sozinho tem uns {int(np.percentile(pts_com, 97.5) - np.percentile(pts_com, 2.5))} pontos de largura.")
print(f"  O efeito aparece na liga inteira, 380 jogos, e some no placar de um clube.")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · a série por temporada ══
fig1, ax1 = plt.subplots(figsize=(12, 6))
anos = tab.index.values
cores = [VERMELHO if t == 2020 else (DOURADO if t == 2021 else VERDE) for t in anos]
ax1.errorbar(anos, tab["p_casa"] * 100,
             yerr=[(tab["p_casa"] - tab["lo"]) * 100, (tab["hi"] - tab["p_casa"]) * 100],
             fmt="none", ecolor="#bbbbbb", elinewidth=1.2, capsize=3, zorder=1)
ax1.scatter(anos, tab["p_casa"] * 100, c=cores, s=70, zorder=3)
ax1.axhline(m * 100, color="#333333", lw=1.2, ls="--")
ax1.text(2024.6, m * 100 + 0.5, f"média\n{pct(m)}", fontsize=10, color="#333333", ha="right", va="bottom")
ax1.annotate("2020, sem público", xy=(2020, tab.loc[2020, "p_casa"] * 100),
             xytext=(2018.5, 56.5), fontsize=11, color=VERMELHO, fontweight="bold",
             arrowprops=dict(arrowstyle="->", color=VERMELHO, lw=1.5))
ax1.text(2021, tab.loc[2021, "p_casa"] * 100 + 1.2, "2021,\ntransição", fontsize=9, color="#8a6f1f", ha="center", va="bottom")
ax1.annotate("2017, com público: 43,9%", xy=(2017, tab.loc[2017, "p_casa"] * 100),
             xytext=(2009.5, 38.5), fontsize=10, color="#333333",
             arrowprops=dict(arrowstyle="->", color="#333333", lw=1.2))
ax1.set_ylim(36, 60)
ax1.set_xticks(anos[::2])
ax1.set_ylabel("jogos vencidos pelo time da casa (%)")
ax1.set_title("O time da casa vence um pouco menos da metade dos jogos\n"
              "Série A do Brasileirão, 2003 a 2024, 8.785 jogos · barras = intervalo de credibilidade de 95%",
              fontsize=13, pad=14)
plt.figtext(0.5, 0.005, "Fonte: resultados oficiais da CBF, base pública · posterior de Jeffreys por temporada"
            "  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-117-grafico-01-por-temporada.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · o experimento natural ══
fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(13.5, 5.8))
xs = np.linspace(0.36, 0.56, 600)
for r, cor, rot in [("com público (2003 a 2019)", VERDE, "com público, 2003 a 2019"),
                    ("sem público (2020)", VERMELHO, "sem público, 2020"),
                    ("público de volta (2022 a 2024)", DOURADO, "público de volta, 2022 a 2024")]:
    ax2a.plot(xs * 100, reg[r]["post"].pdf(xs), color=cor, lw=2.6, label=rot)
    ax2a.fill_between(xs * 100, reg[r]["post"].pdf(xs), color=cor, alpha=0.12)
ax2a.set_xlabel("probabilidade de o time da casa vencer (%)")
ax2a.set_ylabel("densidade")
ax2a.legend(fontsize=10, loc="upper left")
ax2a.set_title("Três regimes, três posteriores", fontsize=12.5, pad=10)

ax2b.hist(dif_ab * 100, bins=80, color=VERMELHO, alpha=0.75, density=True,
          label="2003-2019 menos 2020")
ax2b.hist(dif_ac * 100, bins=80, color=DOURADO, alpha=0.65, density=True,
          label="2003-2019 menos 2022-2024")
ax2b.axvline(0, color="#333333", lw=1.4)
ma, la, ha = ic_sim(dif_ab); mc, lc, hc = ic_sim(dif_ac)
ax2b.text(0.98, 0.95, f"sem público: {vir(ma * 100)} pp [{vir(la * 100)}; {vir(ha * 100)}]\n"
          f"volta do público: {vir(mc * 100)} pp [{vir(lc * 100)}; {vir(hc * 100)}]",
          transform=ax2b.transAxes, ha="right", va="top", fontsize=10.5)
ax2b.set_xlabel("pontos percentuais a menos de vitória em casa")
ax2b.set_ylabel("densidade")
ax2b.legend(fontsize=10, loc="upper left")
ax2b.set_title("Quanto a casa perdeu, e quanto voltou", fontsize=12.5, pad=10)
plt.figtext(0.5, 0.005, "Experimento natural: o Brasileirão de 2020 inteiro sem público · 2021 excluído por ser transição"
            "  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-117-grafico-02-sem-publico.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · a viagem ══
fig3, ax3 = plt.subplots(figsize=(11, 5.8))
nomes = [f[0] for f in faixa_res]; vals = [f[2] * 100 for f in faixa_res]
los = [(f[2] - f[3]) * 100 for f in faixa_res]; his = [(f[4] - f[2]) * 100 for f in faixa_res]
ns = [f[1] for f in faixa_res]
ax3.bar(nomes, vals, color=[CINZA, VERDE, VERDE, VERDE], alpha=0.85, width=0.55)
ax3.errorbar(nomes, vals, yerr=[los, his], fmt="none", ecolor="#222222", elinewidth=1.6, capsize=6)
for i, (v, n, h) in enumerate(zip(vals, ns, his)):
    ax3.text(i, v + h + 0.5, f"{vir(v)}%", ha="center", fontsize=13, fontweight="bold")
    ax3.text(i, 38.7, f"{mil(n)} jogos", ha="center", fontsize=9.5, color="white")
ax3.axhline(m * 100, color="#333333", lw=1.2, ls="--")
ax3.set_ylim(38, 60)
ax3.set_ylabel("jogos vencidos pelo time da casa (%)")
ax3.set_title(f"Quanto mais longe vem o visitante, mais a casa vence: {vir(b1, 3)} gol de saldo por 1.000 km\n"
              "distância entre capitais dos estados como aproximação · Pollard 2008 achou 0,115 com as cidades",
              fontsize=12.5, pad=12)
plt.figtext(0.5, 0.005, "Fonte: base pública da CBF, 2003 a 2024 · barras de erro = intervalo de credibilidade de 95%"
            "  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-117-grafico-03-a-viagem.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · a temporada ══
fig4, ax4 = plt.subplots(figsize=(11, 5.8))
bins = np.arange(10, 50) - 0.5
ax4.hist(pts_com, bins=bins, color=VERDE, alpha=0.6, density=True, label="com público, 2003 a 2019")
ax4.hist(pts_sem, bins=bins, color=VERMELHO, alpha=0.55, density=True, label="sem público, 2020")
ax4.axvline(pts_com.mean(), color=VERDE, lw=2)
ax4.axvline(pts_sem.mean(), color=VERMELHO, lw=2)
ax4.text(0.98, 0.95, f"médias: {vir(pts_com.mean())} contra {vir(pts_sem.mean())} pontos\n"
         f"diferença de {vir(dif_temp.mean())} pontos numa temporada\n"
         f"largura do acaso de um clube: ~{int(np.percentile(pts_com, 97.5) - np.percentile(pts_com, 2.5))} pontos",
         transform=ax4.transAxes, ha="right", va="top", fontsize=10.5)
ax4.set_xlabel("pontos conquistados em casa por um clube médio, 19 jogos")
ax4.set_ylabel("densidade")
ax4.legend(fontsize=10, loc="upper left")
ax4.set_title("Sem público, um clube médio perde menos de três pontos por temporada; o acaso vale vinte\n"
              f"{mil(N_SIM)} temporadas sorteadas por regime, semente 42",
              fontsize=12.5, pad=12)
plt.figtext(0.5, 0.005, "Simulação própria sobre as proporções observadas (Dirichlet de Jeffreys) · clube médio, sem força relativa"
            "  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-117-grafico-04-a-temporada.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**O time da casa vence 49,6% dos jogos.** Em 8.785 partidas de 2003 a 2024, com um
intervalo de dois pontos de largura. Empata 26,4%. O visitante leva 24,0%.

Não é "esse a gente ganha". É uma moeda com um lado um pouco mais pesado.

Em 2020, com os estádios vazios a temporada inteira, a vitória em casa ficou em 45,0%.
Contra os dezessete anos com público, a diferença é de 5,7 pontos percentuais, e a
posterior deixa 1,6% de massa do lado do zero. Contra os três anos imediatamente
anteriores, a diferença cai para 3,5 pontos, com mais dúvida. O tamanho depende da régua.
A direção, não.

E a parte que eu não esperava: o público voltou, e a vantagem não voltou. De 2022 a
2024, o mandante venceu 46,1%, quase o mesmo que no ano vazio e 4,5 pontos abaixo da
média histórica. Metade da queda aconteceu antes da pandemia: 2017 teve estádio cheio e
43,9%. A casa vem valendo menos há quase dez anos, e a torcida é uma parte disso, não a
explicação inteira.

A viagem também é uma parte pequena. Cada mil quilômetros que o visitante percorre valem
cerca de sete centésimos de gol de saldo, a mesma ordem de grandeza que o artigo de 2008
mediu com outra régua. E nos clássicos do mesmo estado, em que ninguém viaja, o mandante
vence menos que a média: 44,8%.

Numa temporada, para um clube, tudo isso vale menos de três pontos. O acaso de 19 jogos
em casa vale vinte. O efeito existe na liga inteira, em 380 jogos, e desaparece dentro
do campeonato de um clube só. É por isso que a sensação de "em casa a gente ganha"
sobrevive a tantos 1 a 1.

*Quantas vezes você já deu um jogo como ganho pelo lugar onde ele ia acontecer?*

---

### ⚠️ Limitações do Modelo

- A base é pública e compilada dos registros da CBF; foi conferida por temporada,
  data, placar e coerência de vencedor, e reproduz o 65,2% de pontos do mandante que
  Pollard, Silva & Medeiros 2008 publicaram para 2003 a 2007. Não é a base oficial.
- A série termina em 2024. As temporadas de 2025 e 2026 não estão incluídas.
- 2020 sem público é um experimento natural, não um sorteio: a pandemia mudou
  calendário, elenco e preparação ao mesmo tempo. A diferença medida é "2020 contra os
  outros anos", e a torcida é a explicação mais plausível, não a única. O tamanho do
  efeito varia de 3,5 a 5,7 pontos conforme a linha de base, e a vantagem da casa já
  vinha caindo desde 2017: parte do que 2020 mostra é tendência, não ausência de público.
- 2021 foi excluído das comparações: o público voltou estádio a estádio, e a base não
  registra quem jogou com e quem jogou sem torcida.
- A distância usa as capitais dos estados, não as cidades dos clubes; clássicos
  estaduais entram com zero. O coeficiente é comparável ao de Pollard em ordem de
  grandeza, não ponto a ponto.
- O modelo trata os jogos como independentes e o "clube médio" não tem força relativa:
  a simulação mede a vantagem da casa, não a chance de um clube específico.
- Bryson et al. 2021 não acharam efeito significativo em vitórias nas ligas europeias
  de 2019/20, com um terço de temporada sem público; aqui a temporada inteira foi
  vazia. Os dois resultados não se contradizem: medem exposições diferentes.
- 2003 (24 clubes) e 2005 (22 clubes) têm fórmula diferente; 2016 tem 379 jogos.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
